<a href="https://colab.research.google.com/github/Luca4Spreafico/CHALLENGE-2---Ibuprofen/blob/main/Swin_%2B_CNN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Swin Transformer

# Google colab:

In [6]:
from google.colab import drive

import os
import shutil
import numpy as np
from PIL import Image
import pandas as pd
from pathlib import Path
from tqdm import tqdm

!pip install lion-pytorch
from lion_pytorch import Lion
!pip install ranger21
from ranger21 import Ranger21
# Define your working directory
drive.mount("/gdrive")
working_dir = "/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2"
%cd $working_dir



Drive already mounted at /gdrive; to attempt to forcibly remount, call drive.mount("/gdrive", force_remount=True).
/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2


# Cropping for Training

In [ ]:
from google.colab import drive
import os
import numpy as np
from PIL import Image
from tqdm import tqdm
import cv2

# Mount and navigate
drive.mount("/gdrive")
working_dir = "/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2"
%cd $working_dir

def optimal_fixed_size_tiles(
    image,
    mask,
    tile_size,
    overlap_pixels=0,
    mask_pixel_threshold=0.0,
    background_value=0.0,
):
    """
    Generate fixed-size square tiles (tile_size x tile_size) that best cover
    a given mask, using a greedy selection of tile positions + fallback for
    isolated leftovers.

    Then deduplicate tiles that cover exactly the same set of mask pixels,
    keeping only the best-centered tile in each group.

    Args:
        image (np.ndarray): Original RGB image, float32 [0,1], shape (H, W, C).
        mask (np.ndarray): Binary mask (0/1, 0/255, bool), shape (H, W).
        tile_size (int): Size of the square crops.
        overlap_pixels (int): Overlap between neighboring candidate tiles.
                              Step = tile_size - overlap_pixels.
        mask_pixel_threshold (float): Minimum fraction of mask pixels inside a tile
                                      (relative to tile_size * tile_size) to even
                                      consider that tile in the greedy stage.
        background_value (float): Value to pad outside-image regions with.

    Returns:
        list of tuples:
            (image_tile, mask_tile, (row_start, col_start))
    """

    H, W = image.shape[:2]

    step = tile_size - overlap_pixels
    if step <= 0:
        raise ValueError("overlap_pixels must be < tile_size (step must be > 0).")

    mask_bin = (mask > 0).astype(np.uint8)

    if mask_bin.sum() == 0:
        return []

    if tile_size >= H:
        r_starts = [0]
    else:
        r_starts = list(range(0, H - tile_size + 1, step))
        if r_starts[-1] != H - tile_size:
            r_starts.append(H - tile_size)

    if tile_size >= W:
        c_starts = [0]
    else:
        c_starts = list(range(0, W - tile_size + 1, step))
        if c_starts[-1] != W - tile_size:
            c_starts.append(W - tile_size)

    candidate_tiles = []
    tile_area = float(tile_size * tile_size)

    for r in r_starts:
        for c in c_starts:
            copy_r_start = max(0, r)
            copy_r_end   = min(H, r + tile_size)
            copy_c_start = max(0, c)
            copy_c_end   = min(W, c + tile_size)

            if copy_r_start >= copy_r_end or copy_c_start >= copy_c_end:
                continue

            tile_mask = mask_bin[copy_r_start:copy_r_end, copy_c_start:copy_c_end]
            mask_pixels = tile_mask.sum()
            frac_mask = mask_pixels / tile_area

            if mask_pixels > 0 and frac_mask >= mask_pixel_threshold:
                candidate_tiles.append((mask_pixels, r, c))

    tiles = []
    selected_positions = set()
    uncovered = mask_bin.copy()

    if candidate_tiles:
        candidate_tiles.sort(key=lambda x: x[0], reverse=True)

        for mask_pixels, r, c in candidate_tiles:
            copy_r_start = max(0, r)
            copy_r_end   = min(H, r + tile_size)
            copy_c_start = max(0, c)
            copy_c_end   = min(W, c + tile_size)

            tile_uncovered = uncovered[copy_r_start:copy_r_end, copy_c_start:copy_c_end]
            new_coverage = tile_uncovered.sum()

            if new_coverage == 0:
                continue

            img_tile = np.full(
                (tile_size, tile_size, image.shape[2]),
                background_value,
                dtype=image.dtype
            )
            mask_tile = np.zeros((tile_size, tile_size), dtype=mask.dtype)

            paste_r_start = copy_r_start - r
            paste_c_start = copy_c_start - c
            paste_r_end   = paste_r_start + (copy_r_end - copy_r_start)
            paste_c_end   = paste_c_start + (copy_c_end - copy_c_start)

            img_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end, :] = \
                image[copy_r_start:copy_r_end, copy_c_start:copy_c_end, :]
            mask_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end] = \
                mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end]

            tiles.append((img_tile, mask_tile, (r, c)))
            selected_positions.add((r, c))

            uncovered[copy_r_start:copy_r_end, copy_c_start:copy_c_end] = 0

            if uncovered.sum() == 0:
                break

    if uncovered.sum() > 0:
        num_labels, labels = cv2.connectedComponents(uncovered.astype(np.uint8))

        for label_id in range(1, num_labels):
            ys, xs = np.where(labels == label_id)
            if ys.size == 0:
                continue

            center_r = int(ys.mean())
            center_c = int(xs.mean())

            if tile_size >= H:
                r_forced = 0
            else:
                r_forced = center_r - tile_size // 2
                r_forced = max(0, min(r_forced, H - tile_size))

            if tile_size >= W:
                c_forced = 0
            else:
                c_forced = center_c - tile_size // 2
                c_forced = max(0, min(c_forced, W - tile_size))

            if (r_forced, c_forced) in selected_positions:
                continue

            copy_r_start = max(0, r_forced)
            copy_r_end   = min(H, r_forced + tile_size)
            copy_c_start = max(0, c_forced)
            copy_c_end   = min(W, c_forced + tile_size)

            img_tile = np.full(
                (tile_size, tile_size, image.shape[2]),
                background_value,
                dtype=image.dtype
            )
            mask_tile = np.zeros((tile_size, tile_size), dtype=mask.dtype)

            paste_r_start = copy_r_start - r_forced
            paste_c_start = copy_c_start - c_forced
            paste_r_end   = paste_r_start + (copy_r_end - copy_r_start)
            paste_c_end   = paste_c_start + (copy_c_end - copy_c_start)

            img_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end, :] = \
                image[copy_r_start:copy_r_end, copy_c_start:copy_c_end, :]
            mask_tile[paste_r_start:paste_r_end, paste_c_start:paste_c_end] = \
                mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end]

            tiles.append((img_tile, mask_tile, (r_forced, c_forced)))
            selected_positions.add((r_forced, c_forced))

    if not tiles:
        return []

    dedup_map = {}
    for idx, (_, _, (r, c)) in enumerate(tiles):
        copy_r_start = max(0, r)
        copy_r_end   = min(H, r + tile_size)
        copy_c_start = max(0, c)
        copy_c_end   = min(W, c + tile_size)

        tile_mask = (mask[copy_r_start:copy_r_end, copy_c_start:copy_c_end] > 0)
        ys, xs = np.where(tile_mask)

        if ys.size == 0:
            global_idxs = ()
            centroid_r = centroid_c = None
        else:
            global_ys = ys + copy_r_start
            global_xs = xs + copy_c_start
            global_idxs = tuple(sorted(global_ys * W + global_xs))

            centroid_r = float(global_ys.mean())
            centroid_c = float(global_xs.mean())

        tile_center_r = r + tile_size / 2.0
        tile_center_c = c + tile_size / 2.0

        if centroid_r is None:
            score = 0.0
        else:
            dr = tile_center_r - centroid_r
            dc = tile_center_c - centroid_c
            score = dr * dr + dc * dc

        if global_idxs not in dedup_map or score < dedup_map[global_idxs][0]:
            dedup_map[global_idxs] = (score, idx)

    final_tiles = [tiles[v[1]] for v in dedup_map.values()]

    return final_tiles


def crop_train_images_optimal(input_dir, input_mask_dir, output_dir,
                               tile_size=144, overlap=64):
    """
    Use the optimal tiling function to crop training images.
    """
    print("="*60)
    print("OPTIMAL CROPPING - MINIMAL TILES TO COVER TISSUE")
    print("="*60)

    os.makedirs(output_dir, exist_ok=True)

    image_files = sorted([f for f in os.listdir(input_dir)
                         if f.lower().endswith('.png')])

    print(f"Found {len(image_files)} training images")
    print(f"Tile size: {tile_size}x{tile_size}, Overlap: {overlap}px")

    total_crops = 0

    for img_name in tqdm(image_files, desc="Optimal cropping"):
        # Load image
        img_path = os.path.join(input_dir, img_name)
        image_pil = Image.open(img_path).convert('RGB')
        # Convert to numpy array [0-1] float32 (required by function)
        image = np.array(image_pil).astype(np.float32) / 255.0

        # Load mask
        mask_name = img_name.replace('img_', 'mask_')
        mask_path = os.path.join(input_mask_dir, mask_name)

        if not os.path.exists(mask_path):
            print(f"Warning: No mask for {img_name}")
            continue

        mask_pil = Image.open(mask_path).convert('L')
        mask = np.array(mask_pil)  # Binary mask (0 or 255)

        # Call the optimal tiling function
        tiles = optimal_fixed_size_tiles(
            image=image,
            mask=mask,
            tile_size=tile_size,
            overlap_pixels=overlap,
            mask_pixel_threshold=0.01,  # Require at least 1% mask coverage
            background_value=0.0  # Pad with black if needed
        )

        # Save the tiles
        base_name = os.path.splitext(img_name)[0]

        for crop_idx, (img_tile, mask_tile, (r, c)) in enumerate(tiles):
            # Convert back to [0-255] uint8 for saving
            img_tile_uint8 = (img_tile * 255).astype(np.uint8)
            img_tile_pil = Image.fromarray(img_tile_uint8)

            crop_name = f"{base_name}_crop{crop_idx:03d}.png"
            img_tile_pil.save(os.path.join(output_dir, crop_name))
            total_crops += 1

    print(f"\n✓ Created {total_crops} crops from {len(image_files)} images")
    print(f"✓ Saved to: {output_dir}")
    return total_crops

# Run optimal cropping
if __name__ == '__main__':
    crop_train_images_optimal(
        input_dir='organized_data/test_144_masked_images',
        input_mask_dir='organized_data/test_masks',
        output_dir='test_crops_144_ov64',
        tile_size=144,
        overlap=64
    )

In [2]:
import os

# Define the directory
test_dir = 'test_crops_144_ov64'

# Get all files
files = [f for f in os.listdir(test_dir) if f.endswith('.png')]

# Rename each file
for old_name in files:
    # img_0000_crop000.png -> img_0000_000.png
    new_name = old_name.replace('_crop', '_')

    old_path = os.path.join(test_dir, old_name)
    new_path = os.path.join(test_dir, new_name)

    os.rename(old_path, new_path)

# Create labels cropping

In [ ]:
from google.colab import drive

import os
import shutil
import numpy as np
from PIL import Image
import pandas as pd
from pathlib import Path
from tqdm import tqdm

!pip install lion-pytorch

# Define your working directory
drive.mount("/gdrive")
working_dir = "/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2"
%cd $working_dir
import pandas as pd
import os

def create_crop_labels(original_labels_path, crops_dir, output_labels_path):
    """
    Create a labels file for cropped images.
    Each crop inherits the label from its parent image.

    Args:
        original_labels_path: Path to train_labels_clean.csv
        crops_dir: Directory containing cropped images (train_crops_384_ov128)
        output_labels_path: Where to save the new labels CSV
    """
    print("="*60)
    print("CREATING LABELS FOR CROPPED IMAGES")
    print("="*60)

    # Load original labels
    original_labels = pd.read_csv(original_labels_path)
    print(f"Loaded {len(original_labels)} original labels")
    print(f"Columns: {list(original_labels.columns)}")

    # Get all cropped images
    crop_files = sorted([f for f in os.listdir(crops_dir)
                        if f.lower().endswith('.png')])
    print(f"Found {len(crop_files)} cropped images")

    # Create new labels
    new_labels = []

    for crop_name in crop_files:
        # Extract original image filename from crop name
        # Example: "img_0042_crop003.png" -> "img_0042.png"
        parts = crop_name.split('_crop')
        original_filename = parts[0] + '.png'  # "img_0042.png"

        # Find the label for this original image
        label_row = original_labels[original_labels['sample_index'] == original_filename]

        if len(label_row) == 0:
            print(f"Warning: No label found for {original_filename}")
            continue

        label = label_row['label'].values[0]  # ← Line 42: Changed 'label' to 'labels'

        # Add entry for this crop
        new_labels.append({
            'sample_index': crop_name,
            'label': label
        })

    # Create DataFrame and save
    labels_df = pd.DataFrame(new_labels)
    labels_df.to_csv(output_labels_path, index=False)

    print("\n" + "="*60)
    print("LABELS CREATION COMPLETE")
    print("="*60)
    print(f"✓ Created labels for {len(labels_df)} crops")
    print(f"✓ Saved to: {output_labels_path}")
    print(f"\nLabel distribution:")
    print(labels_df['label'].value_counts().sort_index())  # ← Line 62: Changed 'labels' to 'label'
    print("="*60)

    return labels_df

# Run the label creation
if __name__ == '__main__':
    labels_df = create_crop_labels(
        original_labels_path='train_labels_clean.csv',
        crops_dir='train_crops_384_ov128',
        output_labels_path='train_labels_crops_crop384_ov128.csv'
    )

    # Preview
    print("\nFirst few entries:")
    print(labels_df.head(10))

Mounted at /gdrive
/gdrive/My Drive/B University/Artificial Networks/an2dl2526c2
CREATING LABELS FOR CROPPED IMAGES
Loaded 581 original labels
Columns: ['sample_index', 'label']
Found 2179 cropped images

LABELS CREATION COMPLETE
✓ Created labels for 2179 crops
✓ Saved to: train_labels_crops_crop384_ov128.csv

Label distribution:
label
HER2(+)            570
Luminal A          593
Luminal B          776
Triple negative    240
Name: count, dtype: int64

First few entries:
           sample_index            label
0  img_0000_crop000.png  Triple negative
1  img_0000_crop001.png  Triple negative
2  img_0002_crop000.png        Luminal B
3  img_0002_crop001.png        Luminal B
4  img_0002_crop002.png        Luminal B
5  img_0002_crop003.png        Luminal B
6  img_0002_crop004.png        Luminal B
7  img_0003_crop000.png        Luminal B
8  img_0004_crop000.png        Luminal B
9  img_0004_crop001.png        Luminal B


# Unzip folder

In [ ]:
!unzip train_data_crops_crop512_ov128.zip

Archive:  train_data_crops_crop512_ov128.zip
  inflating: train_data_crops_crop512_ov128/img_0687_000.png  
replace train_data_crops_crop512_ov128/mask_0690_001.png? [y]es, [n]o, [A]ll, [N]one, [r]ename: N


# Train & Val

In [11]:
"""
Dual-Path Multi-Scale Breast Cancer Classification
- Swin Transformer: 512×512 full images (global context)
- CNN: 144×144 masked tissue (fine details)
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Label mapping
LABEL_MAP = {
    'Luminal A': 0,
    'Luminal B': 1,
    'HER2(+)': 2,
    'Triple negative': 3
}

LABEL_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']


class DualScaleDataset(Dataset):
    """
    Dataset that loads BOTH 512×512 and 144×144 crops
    Handles different naming conventions:
    - 512 crops: img_0000_000.png
    - 144 crops: img_0000_crop000.png
    """
    def __init__(self, csv_512, img_dir_512, csv_144, img_dir_144,
                 transform_512=None, transform_144=None, label_col='label'):
        """
        Args:
            csv_512: CSV for 512×512 crops
            img_dir_512: Directory for 512×512 images
            csv_144: CSV for 144×144 crops
            img_dir_144: Directory for 144×144 images
            transform_512: Transforms for Swin (512 images)
            transform_144: Transforms for CNN (144 images)
        """
        # Load both CSVs
        self.df_512 = pd.read_csv(csv_512)
        self.df_144 = pd.read_csv(csv_144)

        self.img_dir_512 = img_dir_512
        self.img_dir_144 = img_dir_144
        self.transform_512 = transform_512
        self.transform_144 = transform_144
        self.label_col = label_col

        # Map labels to integers
        if self.df_512[label_col].dtype == 'object':
            self.df_512['label_int'] = self.df_512[label_col].map(LABEL_MAP)
            self.df_144['label_int'] = self.df_144[label_col].map(LABEL_MAP)
        else:
            self.df_512['label_int'] = self.df_512[label_col]
            self.df_144['label_int'] = self.df_144[label_col]

        # Extract original image IDs for matching
        # 512: img_0000_000.png → img_0000
        # 144: img_0000_crop000.png → img_0000
        self.df_512['orig_img_id'] = self.df_512['sample_index'].str.extract(r'(img_\d+)')[0]
        self.df_144['orig_img_id'] = self.df_144['sample_index'].str.extract(r'(img_\d+)')[0]

        # Extract crop numbers for matching
        # 512: img_0000_000.png → 000
        # 144: img_0000_crop000.png → 000
        self.df_512['crop_num'] = self.df_512['sample_index'].str.extract(r'_(\d{3})\.png')[0]
        self.df_144['crop_num'] = self.df_144['sample_index'].str.extract(r'crop(\d{3})\.png')[0]

        # Create matching keys: img_0000_000
        self.df_512['match_key'] = self.df_512['orig_img_id'] + '_' + self.df_512['crop_num']
        self.df_144['match_key'] = self.df_144['orig_img_id'] + '_' + self.df_144['crop_num']

        # Find common crops (crops that exist in BOTH 512 and 144 datasets)
        common_keys = set(self.df_512['match_key']).intersection(set(self.df_144['match_key']))

        # Filter to only keep matching crops
        self.df_512 = self.df_512[self.df_512['match_key'].isin(common_keys)].reset_index(drop=True)
        self.df_144 = self.df_144[self.df_144['match_key'].isin(common_keys)].reset_index(drop=True)

        # Sort both by match_key to ensure alignment
        self.df_512 = self.df_512.sort_values('match_key').reset_index(drop=True)
        self.df_144 = self.df_144.sort_values('match_key').reset_index(drop=True)

        # Verify alignment
        assert all(self.df_512['match_key'] == self.df_144['match_key']), "Crop alignment failed!"
        assert all(self.df_512['label_int'] == self.df_144['label_int']), "Label mismatch!"

        print(f"Loaded {len(self.df_512)} matching crop pairs")
        print(f"Class distribution:\n{self.df_512[label_col].value_counts()}")

    def __len__(self):
        return len(self.df_512)

    def __getitem__(self, idx):
        # Get filenames and label
        img_name_512 = self.df_512.iloc[idx]['sample_index']
        img_name_144 = self.df_144.iloc[idx]['sample_index']
        label = self.df_512.iloc[idx]['label_int']
        match_key = self.df_512.iloc[idx]['match_key']

        # Load 512×512 image
        img_path_512 = os.path.join(self.img_dir_512, img_name_512)
        image_512 = Image.open(img_path_512).convert('RGB')

        # Load 144×144 image
        img_path_144 = os.path.join(self.img_dir_144, img_name_144)
        image_144 = Image.open(img_path_144).convert('RGB')

        # Apply transforms
        if self.transform_512:
            image_512 = self.transform_512(image_512)
        if self.transform_144:
            image_144 = self.transform_144(image_144)

        return image_512, image_144, label, match_key


class MultiScaleDualPathClassifier(nn.Module):
    """
    Dual-path classifier:
    - Swin Transformer for 512×512 images (global context)
    - CNN for 144×144 masked images (fine details)
    """
    def __init__(self, num_classes=4, pretrained=True):
        super(MultiScaleDualPathClassifier, self).__init__()

        # ===== FINE-DETAIL PATH: CNN for 144×144 masked tissue =====
        self.cnn_backbone = nn.Sequential(
            # Input: [B, 3, 144, 144]
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),  # → [B, 64, 72, 72]
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),     # → [B, 64, 36, 36]

            self._make_resnet_block(64, 128, 2),   # → [B, 128, 18, 18]
            self._make_resnet_block(128, 256, 2),  # → [B, 256, 9, 9]
            self._make_resnet_block(256, 512, 2),  # → [B, 512, 5, 5]

            nn.AdaptiveAvgPool2d((1, 1)),  # → [B, 512, 1, 1]
            nn.Flatten()                   # → [B, 512]
        )

        # ===== GLOBAL CONTEXT PATH: Swin for 512×512 full images =====
        self.swin_backbone = timm.create_model(
            'swin_base_patch4_window12_384',
            pretrained=pretrained,
            num_classes=0,      # Remove classification head
            global_pool='avg'   # Global average pooling
        )

        # Feature dimensions
        cnn_feature_dim = 512
        swin_feature_dim = self.swin_backbone.num_features  # 1024 for swin_base

        # ===== FUSION LAYER =====
        # Combines features from both paths
       # NEW (simpler):
        self.fusion = nn.Sequential(
            nn.Linear(cnn_feature_dim + swin_feature_dim, 768),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

        print(f"✓ CNN path: 144×144 → {cnn_feature_dim} features (fine tissue details)")
        print(f"✓ Swin path: 512×512 → {swin_feature_dim} features (global context)")
        print(f"✓ Fusion: {cnn_feature_dim + swin_feature_dim} → {num_classes} classes")

    def _make_resnet_block(self, in_channels, out_channels, num_blocks):
        """ResNet-style convolutional block"""
        layers = []

        # First block with stride=2 for downsampling
        layers.append(nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))

        # Remaining blocks maintain resolution
        for _ in range(num_blocks - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, 3, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))

        return nn.Sequential(*layers)

    def forward(self, img_512, img_144):
        """
        Forward pass through both paths

        Args:
            img_512: [B, 3, 512, 512] - Full images for Swin
            img_144: [B, 3, 144, 144] - Masked tissue for CNN

        Returns:
            logits: [B, num_classes]
        """
        # CNN PATH: Process 144×144 masked tissue
        cnn_features = self.cnn_backbone(img_144)  # [B, 512]

        # SWIN PATH: Process 512×512 full images
        swin_features = self.swin_backbone(img_512)  # [B, 1024]

        # FUSION: Combine both feature sets
        combined = torch.cat([cnn_features, swin_features], dim=1)  # [B, 1536]
        output = self.fusion(combined)

        return output


def get_transforms(img_size_512=384, img_size_144=144, augment=True):
    """
    Get transforms for both image sizes
    """
    if augment:
        # Swin transform (512 images)
        transform_512 = transforms.Compose([
            transforms.Resize((img_size_512, img_size_512)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        # CNN transform (144 images)
        transform_144 = transforms.Compose([
            transforms.Resize((img_size_144, img_size_144)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomVerticalFlip(p=0.5),
            transforms.RandomRotation(90),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])
    else:
        transform_512 = transforms.Compose([
            transforms.Resize((img_size_512, img_size_512)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

        transform_144 = transforms.Compose([
            transforms.Resize((img_size_144, img_size_144)),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                               std=[0.229, 0.224, 0.225])
        ])

    return transform_512, transform_144


def train_epoch(model, train_loader, criterion, optimizer, device):
    """Train for one epoch with dual-path model"""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    pbar = tqdm(train_loader, desc='Training')

    for img_512, img_144, labels, _ in pbar:
        img_512 = img_512.to(device)
        img_144 = img_144.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        # Forward pass through both paths
        outputs = model(img_512, img_144)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

        pbar.set_postfix({
            'loss': f'{running_loss/len(pbar):.4f}',
            'acc': f'{100*correct/total:.2f}%'
        })

    epoch_acc = correct / total
    return epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate the dual-path model"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(val_loader, desc='Validation')
        for img_512, img_144, labels, _ in pbar:
            img_512 = img_512.to(device)
            img_144 = img_144.to(device)
            labels = labels.to(device)

            outputs = model(img_512, img_144)
            loss = criterion(outputs, labels)

            running_loss += loss.item()
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

            pbar.set_postfix({
                'loss': f'{running_loss/len(pbar):.4f}',
                'acc': f'{100*correct/total:.2f}%'
            })

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = correct / total

    return epoch_loss, epoch_acc, all_preds, all_labels


def plot_confusion_matrix(y_true, y_pred, save_path='confusion_matrix.png'):
    """Plot and save confusion matrix"""
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=LABEL_NAMES,
                yticklabels=LABEL_NAMES)
    plt.title('Confusion Matrix - Dual Path Model')
    plt.ylabel('True Label')
    plt.xlabel('Predicted Label')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Confusion matrix saved to {save_path}")


def plot_training_history(train_accs, val_losses, val_accs, save_path='training_history.png'):
    """Plot training history"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    ax1.plot(val_losses, label='Val Loss', marker='s', color='orange')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy plot
    ax2.plot([acc*100 for acc in train_accs], label='Train Acc', marker='o')
    ax2.plot([acc*100 for acc in val_accs], label='Val Acc', marker='s')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy (%)')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.close()
    print(f"Training history saved to {save_path}")


def main():
    """Main training pipeline for dual-path model"""

    # Configuration
    CONFIG = {
        # 512×512 data (for Swin)
        'csv_file_512': 'train_labels_crops_crop512_ov128 (1).csv',
        'img_dir_512': 'train_data_crops_crop512_ov128',

        # 144×144 data (for CNN)
        'csv_file_144': 'train_labels_crops_crop144_ov64.csv',
        'img_dir_144': 'train_crops_144_ov64',

        # Training params
        'img_size_512': 384,  # Swin input size
        'img_size_144': 144,  # CNN input size
        'batch_size': 16,
        'num_epochs': 15,
        'learning_rate': 1e-4,
        'weight_decay': 1e-4,
        'num_workers': 4,
        'save_dir': 'models_dual_path',
    }

    CONFIG_LION = {
      'learning_rate': 1e-5,      # 10x smaller than AdamW
      'weight_decay': 0.1,         # 10x larger than AdamW
      'beta1': 0.9,                # Default
      'beta2': 0.99,                # Default
    }

    # Create save directory
    os.makedirs(CONFIG['save_dir'], exist_ok=True)

    print("="*60)
    print("Dual-Path Multi-Scale Training Configuration")
    print("="*60)
    for key, value in CONFIG.items():
        print(f"{key}: {value}")
    print("="*60)

    # Get transforms for both scales
    train_transform_512, train_transform_144 = get_transforms(
        img_size_512=CONFIG['img_size_512'],
        img_size_144=CONFIG['img_size_144'],
        augment=True
    )

    val_transform_512, val_transform_144 = get_transforms(
        img_size_512=CONFIG['img_size_512'],
        img_size_144=CONFIG['img_size_144'],
        augment=False
    )

    # Load full dataset
    print("\nLoading dual-scale dataset...")
    full_dataset = DualScaleDataset(
        csv_512=CONFIG['csv_file_512'],
        img_dir_512=CONFIG['img_dir_512'],
        csv_144=CONFIG['csv_file_144'],
        img_dir_144=CONFIG['img_dir_144'],
        transform_512=None,
        transform_144=None,
        label_col='label'
    )

    # Split by original image (using 512 dataframe)
    unique_imgs = full_dataset.df_512.groupby('orig_img_id')['label_int'].first().reset_index()

    train_imgs, val_imgs = train_test_split(
        unique_imgs['orig_img_id'].values,
        test_size=0.2,
        stratify=unique_imgs['label_int'].values,
        random_state=42
    )

    print(f"\nSplit by original images:")
    print(f"Train images: {len(train_imgs)}")
    print(f"Val images: {len(val_imgs)}")

    # Create train and val datasets
    train_dataset = DualScaleDataset(
        csv_512=CONFIG['csv_file_512'],
        img_dir_512=CONFIG['img_dir_512'],
        csv_144=CONFIG['csv_file_144'],
        img_dir_144=CONFIG['img_dir_144'],
        transform_512=train_transform_512,
        transform_144=train_transform_144,
        label_col='label'
    )
    train_dataset.df_512 = full_dataset.df_512[full_dataset.df_512['orig_img_id'].isin(train_imgs)].reset_index(drop=True)
    train_dataset.df_144 = full_dataset.df_144[full_dataset.df_144['orig_img_id'].isin(train_imgs)].reset_index(drop=True)

    val_dataset = DualScaleDataset(
        csv_512=CONFIG['csv_file_512'],
        img_dir_512=CONFIG['img_dir_512'],
        csv_144=CONFIG['csv_file_144'],
        img_dir_144=CONFIG['img_dir_144'],
        transform_512=val_transform_512,
        transform_144=val_transform_144,
        label_col='label'
    )
    val_dataset.df_512 = full_dataset.df_512[full_dataset.df_512['orig_img_id'].isin(val_imgs)].reset_index(drop=True)
    val_dataset.df_144 = full_dataset.df_144[full_dataset.df_144['orig_img_id'].isin(val_imgs)].reset_index(drop=True)

    print(f"\nCrops per split:")
    print(f"Train crops: {len(train_dataset)}")
    print(f"Val crops: {len(val_dataset)}")

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=True,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    # Create dual-path model
    print("\nCreating dual-path model...")
    model = MultiScaleDualPathClassifier(
        num_classes=4,
        pretrained=True
    ).to(device)

    '''
    # Loss and optimizer
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.AdamW(
        model.parameters(),
        lr=CONFIG['learning_rate'],
        weight_decay=CONFIG['weight_decay']
    )

    criterion = nn.CrossEntropyLoss()
    optimizer = Lion(
      model.parameters(),
      lr=CONFIG_LION['learning_rate'],
      weight_decay=CONFIG_LION['weight_decay'],
      betas=(CONFIG_LION['beta1'], CONFIG_LION['beta2'])
    )
    '''
    criterion = nn.CrossEntropyLoss()
    optimizer = Ranger21(
        model.parameters(),
        lr=1e-4,
        weight_decay=1e-4,
        num_epochs=30,
        num_batches_per_epoch=len(train_loader)
    )

    # Learning rate scheduler
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=CONFIG['num_epochs']
    )

    # Training loop
    best_val_f1 = 0.0
    train_accs = []
    val_losses = []
    val_accs = []

    print("\n" + "="*60)
    print("Starting Dual-Path Training")
    print("="*60)

    for epoch in range(CONFIG['num_epochs']):
        print(f"\nEpoch {epoch+1}/{CONFIG['num_epochs']}")
        print("-" * 60)

        # Train
        train_acc = train_epoch(
            model, train_loader, criterion, optimizer, device
        )

        # Validate
        val_loss, val_acc, val_preds, val_labels = validate(
            model, val_loader, criterion, device
        )

        # Update scheduler
        scheduler.step()

        # Store metrics
        train_accs.append(train_acc)
        val_losses.append(val_loss)
        val_accs.append(val_acc)

        # Print epoch summary
        print(f"\nEpoch {epoch+1} Summary:")
        print(f"Train Acc: {train_acc*100:.2f}%")
        print(f"Val Acc: {val_acc*100:.2f}%")
        print(f"Learning Rate: {optimizer.param_groups[0]['lr']:.6f}")

        val_f1_macro = f1_score(val_labels, val_preds, average='macro')
        val_f1_weighted = f1_score(val_labels, val_preds, average='weighted')
        print(f"Val F1-Score (macro): {val_f1_macro:.4f}")
        print(f"Val F1-Score (weighted): {val_f1_weighted:.4f}")

        # Save best model
        if val_f1_weighted > best_val_f1:
            best_val_f1 = val_f1_weighted
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'val_acc': val_acc,
                'val_loss': val_loss,
                'val_f1': val_f1_weighted,
            }, os.path.join(CONFIG['save_dir'], 'best_model_dual_path.pth'))
            print(f"✓ New best model saved! Val F1: {val_f1_weighted:.4f}")

        # Save checkpoint every 5 epochs
        if (epoch + 1) % 5 == 0:
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
            }, os.path.join(CONFIG['save_dir'], f'checkpoint_epoch_{epoch+1}.pth'))

    # Final evaluation
    print("\n" + "="*60)
    print("Training Complete!")
    print("="*60)
    print(f"Best Validation F1-Score: {best_val_f1:.4f}")

    # Load best model for final evaluation
    checkpoint = torch.load(os.path.join(CONFIG['save_dir'], 'best_model_dual_path.pth'))
    model.load_state_dict(checkpoint['model_state_dict'])

    # Final validation
    _, final_acc, final_preds, final_labels = validate(
        model, val_loader, criterion, device
    )

    # Print classification report
    print("\nClassification Report:")
    print(classification_report(
        final_labels,
        final_preds,
        target_names=LABEL_NAMES,
        digits=4
    ))

    # Plot confusion matrix
    plot_confusion_matrix(
        final_labels,
        final_preds,
        save_path=os.path.join(CONFIG['save_dir'], 'confusion_matrix_dual_path.png')
    )

    # Plot training history
    plot_training_history(
        train_accs, val_losses, val_accs,
        save_path=os.path.join(CONFIG['save_dir'], 'training_history_dual_path.png')
    )

    print("\n✓ All results saved to:", CONFIG['save_dir'])


if __name__ == '__main__':
    main()

"""
## How the Two Models Interact:
┌─────────────────────────────────────────────────────────────┐
│                    Training Process                          │
└─────────────────────────────────────────────────────────────┘

1. DATA LOADING (DualScaleDataset):
   ├─ Loads img_0000_000.png (512×512) from train_data_crops_crop512_ov128
   ├─ Loads img_0000_crop000.png (144×144) from train_crops_144_ov64
   └─ Matches them using "img_0000_000" → ensures SAME tissue region

2. FORWARD PASS (MultiScaleDualPathClassifier):
   ┌─── 512×512 image ───┐          ┌─── 144×144 image ───┐
   │                     │          │                     │
   │   Swin Transformer  │          │    CNN Backbone     │
   │   (12 layers deep)  │          │   (ResNet-style)    │
   │                     │          │                     │
   │   Learns:           │          │   Learns:           │
   │   - Tissue layout   │          │   - Cell patterns   │
   │   - Spatial context │          │   - Texture details │
   │   - Architecture    │          │   - Fine morphology │
   │                     │          │                     │
   │   Output: 1024 feat │          │   Output: 512 feat  │
   └──────────┬──────────┘          └──────────┬──────────┘
              │                                │
              └────────────┬───────────────────┘
                           │
                    [Concatenate]
                    1536 features
                           │
                    [Fusion Layer]
                    768 → 256 → 4
                           │
                    4 class logits

3. BACKPROPAGATION:
   Loss
    │
    ├──→ Fusion learns: "Weight CNN vs Swin features"
    ├──→ Swin learns: "Better global tissue patterns"
    └──→ CNN learns: "Better local cell features"

   They train SIMULTANEOUSLY but learn DIFFERENT aspects!

4. INFERENCE:
   Both paths MUST agree on final prediction
   - If Swin says "Luminal A" but CNN says "Triple Negative"
   - Fusion layer resolves conflict based on learned weights

"""

Dual-Path Multi-Scale Training Configuration
csv_file_512: train_labels_crops_crop512_ov128 (1).csv
img_dir_512: train_data_crops_crop512_ov128
csv_file_144: train_labels_crops_crop144_ov64.csv
img_dir_144: train_crops_144_ov64
img_size_512: 384
img_size_144: 144
batch_size: 16
num_epochs: 15
learning_rate: 0.0001
weight_decay: 0.0001
num_workers: 4
save_dir: models_dual_path

Loading dual-scale dataset...
Loaded 1830 matching crop pairs
Class distribution:
label
Luminal B          664
Luminal A          497
HER2(+)            469
Triple negative    200
Name: count, dtype: int64

Split by original images:
Train images: 464
Val images: 117
Loaded 1830 matching crop pairs
Class distribution:
label
Luminal B          664
Luminal A          497
HER2(+)            469
Triple negative    200
Name: count, dtype: int64
Loaded 1830 matching crop pairs
Class distribution:
label
Luminal B          664
Luminal A          497
HER2(+)            469
Triple negative    200
Name: count, dtype: int64



Training:   1%|          | 1/92 [00:01<02:15,  1.49s/it, loss=0.0151, acc=25.00%]

params size saved
total param groups = 1
total params in groups = 361


Validation: 100%|██████████| 23/23 [00:03<00:00,  5.77it/s, loss=1.3593, acc=26.72%]



Epoch 1 Summary:
Train Acc: 28.83%
Val Acc: 26.72%
Learning Rate: 0.000099
Val F1-Score (macro): 0.1276
Val F1-Score (weighted): 0.1452
✓ New best model saved! Val F1: 0.1452

Epoch 2/15
------------------------------------------------------------


...

Validation: 100%|██████████| 23/23 [00:03<00:00,  5.77it/s, loss=1.4038, acc=42.15%]



Epoch 15 Summary:
Train Acc: 65.30%
Val Acc: 42.15%
Learning Rate: 0.000000
Val F1-Score (macro): 0.4095
Val F1-Score (weighted): 0.4147

Training Complete!
Best Validation F1-Score: 0.4294


Validation: 100%|██████████| 23/23 [00:03<00:00,  5.78it/s, loss=1.3456, acc=43.25%]



Classification Report:
                 precision    recall  f1-score   support

      Luminal A     0.3925    0.4286    0.4098        98
      Luminal B     0.4356    0.5182    0.4733       137
        HER2(+)     0.5000    0.3222    0.3919        90
Triple negative     0.4286    0.3947    0.4110        38

       accuracy                         0.4325       363
      macro avg     0.4392    0.4159    0.4215       363
   weighted avg     0.4392    0.4325    0.4294       363

Confusion matrix saved to models_dual_path/confusion_matrix_dual_path.png
Training history saved to models_dual_path/training_history_dual_path.png

✓ All results saved to: models_dual_path


'\n## How the Two Models Interact:\n┌─────────────────────────────────────────────────────────────┐\n│                    Training Process                          │\n└─────────────────────────────────────────────────────────────┘\n\n1. DATA LOADING (DualScaleDataset):\n   ├─ Loads img_0000_000.png (512×512) from train_data_crops_crop512_ov128\n   ├─ Loads img_0000_crop000.png (144×144) from train_crops_144_ov64\n   └─ Matches them using "img_0000_000" → ensures SAME tissue region\n\n2. FORWARD PASS (MultiScaleDualPathClassifier):\n   ┌─── 512×512 image ───┐          ┌─── 144×144 image ───┐\n   │                     │          │                     │\n   │   Swin Transformer  │          │    CNN Backbone     │\n   │   (12 layers deep)  │          │   (ResNet-style)    │\n   │                     │          │                     │\n   │   Learns:           │          │   Learns:           │\n   │   - Tissue layout   │          │   - Cell patterns   │\n   │   - Spatial context │         

# Testing

In [16]:
"""
Test Pipeline for Dual-Path Breast Cancer Classification
Uses EXISTING crops: 512×512 for Swin, 144×144 for CNN
"""

import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import timm
from tqdm import tqdm
from collections import defaultdict

# ============================================================================
# PART 1: DUAL-PATH MODEL (same as training)
# ============================================================================

class MultiScaleDualPathClassifier(nn.Module):
    """Dual-path model (same as training)"""
    def __init__(self, num_classes=4, pretrained=False):
        super(MultiScaleDualPathClassifier, self).__init__()

        # CNN for 144×144
        self.cnn_backbone = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=7, stride=2, padding=3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),

            self._make_resnet_block(64, 128, 2),
            self._make_resnet_block(128, 256, 2),
            self._make_resnet_block(256, 512, 2),

            nn.AdaptiveAvgPool2d((1, 1)),
            nn.Flatten()
        )

        # Swin for 512×512
        self.swin_backbone = timm.create_model(
            'swin_base_patch4_window12_384',
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg'
        )

        cnn_feature_dim = 512
        swin_feature_dim = self.swin_backbone.num_features

        # Fusion (simple concatenation, same as training)
        self.fusion = nn.Sequential(
            nn.Linear(cnn_feature_dim + swin_feature_dim, 768),
            nn.ReLU(inplace=True),
            nn.Dropout(0.3),
            nn.Linear(768, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, num_classes)
        )

    def _make_resnet_block(self, in_channels, out_channels, num_blocks):
        layers = []
        layers.append(nn.Conv2d(in_channels, out_channels, 3, stride=2, padding=1))
        layers.append(nn.BatchNorm2d(out_channels))
        layers.append(nn.ReLU(inplace=True))

        for _ in range(num_blocks - 1):
            layers.append(nn.Conv2d(out_channels, out_channels, 3, padding=1))
            layers.append(nn.BatchNorm2d(out_channels))
            layers.append(nn.ReLU(inplace=True))

        return nn.Sequential(*layers)

    def forward(self, img_512, img_144):
        cnn_features = self.cnn_backbone(img_144)
        swin_features = self.swin_backbone(img_512)

        combined = torch.cat([cnn_features, swin_features], dim=1)
        output = self.fusion(combined)

        return output


# ============================================================================
# PART 2: DUAL-SCALE TEST DATASET
# ============================================================================

class DualScaleTestDataset(Dataset):
    """
    Dataset for test that loads BOTH 512×512 and 144×144 crops
    """
    def __init__(self, crop_dir_512, crop_dir_144, transform_512, transform_144):
        self.crop_dir_512 = crop_dir_512
        self.crop_dir_144 = crop_dir_144
        self.transform_512 = transform_512
        self.transform_144 = transform_144

        # Get all crops from BOTH directories
        crops_512 = set([f for f in os.listdir(crop_dir_512) if f.endswith('.png')])
        crops_144 = set([f for f in os.listdir(crop_dir_144) if f.endswith('.png')])

        # Only keep crops that exist in BOTH
        common_crops = crops_512.intersection(crops_144)
        self.crop_list = sorted(list(common_crops))

        # Extract basenames: img_0000_000.png -> img_0000
        self.basenames = [f.rsplit('_', 1)[0] for f in self.crop_list]

        print(f"✓ Found {len(self.crop_list)} matching crops in both directories")
        if len(common_crops) < len(crops_512):
            print(f"  Warning: {len(crops_512) - len(common_crops)} crops from 512 not in 144")
        if len(common_crops) < len(crops_144):
            print(f"  Warning: {len(crops_144) - len(common_crops)} crops from 144 not in 512")

    def __len__(self):
        return len(self.crop_list)

    def __getitem__(self, idx):
        crop_name = self.crop_list[idx]
        basename = self.basenames[idx]

        # Load 512×512 crop
        path_512 = os.path.join(self.crop_dir_512, crop_name)
        img_512 = Image.open(path_512).convert('RGB')

        # Load 144×144 crop
        path_144 = os.path.join(self.crop_dir_144, crop_name)
        img_144 = Image.open(path_144).convert('RGB')

        # Apply transforms
        if self.transform_512:
            img_512 = self.transform_512(img_512)
        if self.transform_144:
            img_144 = self.transform_144(img_144)

        return img_512, img_144, crop_name, basename


# ============================================================================
# PART 3: PREDICTION AND AGGREGATION
# ============================================================================

def predict_crops(model, loader, device):
    """Predict all crops using dual-path model"""
    model.eval()
    predictions = []

    with torch.no_grad():
        for img_512, img_144, crop_names, basenames in tqdm(loader, desc="Predicting"):
            img_512 = img_512.to(device)
            img_144 = img_144.to(device)

            # Dual-path forward
            outputs = model(img_512, img_144)
            probs = torch.softmax(outputs, dim=1)

            for prob, crop_name, basename in zip(probs, crop_names, basenames):
                predictions.append({
                    'crop_filename': crop_name,
                    'original_basename': basename,
                    'probabilities': prob.cpu().numpy()
                })

    return predictions


def aggregate_predictions(predictions, label_names):
    """Aggregate crop predictions to image-level"""
    print("\n" + "="*60)
    print("AGGREGATING PREDICTIONS")
    print("="*60)

    # Group by original image
    image_groups = defaultdict(list)
    for pred in predictions:
        image_groups[pred['original_basename']].append(pred['probabilities'])

    results = []
    for img_basename, probs_list in image_groups.items():
        # Average probabilities across all crops
        avg_probs = np.mean(probs_list, axis=0)
        pred_class = np.argmax(avg_probs)
        confidence = avg_probs[pred_class]

        results.append({
            'sample_index': img_basename + '.png',
            'label': label_names[pred_class],
            'confidence': confidence,
            'num_crops': len(probs_list)
        })

    results_df = pd.DataFrame(results)

    print(f"Aggregated {len(results_df)} images")
    print(f"\nPrediction distribution:")
    print(results_df['label'].value_counts())
    print(f"\nAverage confidence: {results_df['confidence'].mean():.4f}")
    print(f"Average crops per image: {results_df['num_crops'].mean():.1f}")

    return results_df


def save_submission(results_df, output_path):
    """Save submission file"""
    submission = results_df[['sample_index', 'label']].copy()
    submission.to_csv(output_path, index=False)
    print(f"\n✓ Submission saved to: {output_path}")
    return submission


# ============================================================================
# MAIN PIPELINE
# ============================================================================

def main():
    """Complete dual-path test pipeline"""

    # Configuration
    CONFIG = {
        'test_swin_512_dir': 'organized_data/test_crops_512_ov128',         # 512×512 crops for Swin
        'test_cnn_144_dir': 'test_crops_144_ov64',           # 144×144 crops for CNN
        'model_path': 'models_dual_path/best_model_dual_path.pth',
        'submission_path': 'models_dual_path/submission.csv',
        'batch_size': 16,
        'num_workers': 4,
        'img_size_512': 384,  # Swin input size
        'img_size_144': 144,  # CNN input size
    }

    LABEL_NAMES = ['Luminal A', 'Luminal B', 'HER2(+)', 'Triple negative']

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Using device: {device}\n")

    # ========================================================================
    # STEP 1: Load dual-path model
    # ========================================================================
    print("="*60)
    print("STEP 1: LOADING DUAL-PATH MODEL")
    print("="*60)

    model = MultiScaleDualPathClassifier(num_classes=4, pretrained=False).to(device)
    checkpoint = torch.load(CONFIG['model_path'], map_location=device)
    model.load_state_dict(checkpoint['model_state_dict'])
    model.eval()

    print(f"✓ Model loaded from: {CONFIG['model_path']}")
    if 'val_f1' in checkpoint:
        print(f"  Training val F1: {checkpoint['val_f1']:.4f}")
    elif 'val_acc' in checkpoint:
        print(f"  Training val accuracy: {checkpoint['val_acc']*100:.2f}%")

    # ========================================================================
    # STEP 2: Prepare dual-scale dataset
    # ========================================================================
    print("\n" + "="*60)
    print("STEP 2: PREPARING DUAL-SCALE TEST DATA")
    print("="*60)

    # Transforms for 512 (Swin path)
    transform_512 = transforms.Compose([
        transforms.Resize((CONFIG['img_size_512'], CONFIG['img_size_512'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    # Transforms for 144 (CNN path)
    transform_144 = transforms.Compose([
        transforms.Resize((CONFIG['img_size_144'], CONFIG['img_size_144'])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                           std=[0.229, 0.224, 0.225])
    ])

    # Create dual-scale dataset
    dataset = DualScaleTestDataset(
        crop_dir_512=CONFIG['test_swin_512_dir'],
        crop_dir_144=CONFIG['test_cnn_144_dir'],
        transform_512=transform_512,
        transform_144=transform_144
    )

    loader = DataLoader(
        dataset,
        batch_size=CONFIG['batch_size'],
        shuffle=False,
        num_workers=CONFIG['num_workers'],
        pin_memory=True
    )

    print(f"✓ Prepared {len(dataset)} crop pairs for prediction")

    # ========================================================================
    # STEP 3: Predict all crops
    # ========================================================================
    print("\n" + "="*60)
    print("STEP 3: MAKING DUAL-PATH PREDICTIONS")
    print("="*60)

    predictions = predict_crops(model, loader, device)
    print(f"✓ Predicted {len(predictions)} crops")

    # ========================================================================
    # STEP 4: Aggregate and save
    # ========================================================================
    results_df = aggregate_predictions(predictions, LABEL_NAMES)
    submission = save_submission(results_df, CONFIG['submission_path'])

    # Show sample results
    print("\n" + "="*60)
    print("SAMPLE PREDICTIONS")
    print("="*60)
    print(submission.head(10).to_string(index=False))

    print("\n" + "="*60)
    print("✓ DUAL-PATH TESTING COMPLETE!")
    print("="*60)
    print(f"✓ Used 512 crops from: {CONFIG['test_swin_512_dir']}")
    print(f"✓ Used 144 crops from: {CONFIG['test_cnn_144_dir']}")
    print(f"✓ Submission saved: {CONFIG['submission_path']}")
    print(f"✓ Total predictions: {len(submission)}")


if __name__ == '__main__':
    main()

Using device: cuda

STEP 1: LOADING DUAL-PATH MODEL
✓ Model loaded from: models_dual_path/best_model_dual_path.pth
  Training val F1: 0.4294

STEP 2: PREPARING DUAL-SCALE TEST DATA
✓ Found 1519 matching crops in both directories
✓ Prepared 1519 crop pairs for prediction

STEP 3: MAKING DUAL-PATH PREDICTIONS


Predicting: 100%|██████████| 95/95 [03:59<00:00,  2.52s/it]


✓ Predicted 1519 crops

AGGREGATING PREDICTIONS
Aggregated 477 images

Prediction distribution:
label
Luminal B          236
Luminal A          133
HER2(+)             76
Triple negative     32
Name: count, dtype: int64

Average confidence: 0.5702
Average crops per image: 3.2

✓ Submission saved to: models_dual_path/submission.csv

SAMPLE PREDICTIONS
sample_index     label
img_0000.png Luminal A
img_0001.png Luminal A
img_0002.png Luminal A
img_0003.png Luminal B
img_0004.png Luminal B
img_0005.png   HER2(+)
img_0006.png Luminal B
img_0007.png Luminal B
img_0008.png Luminal B
img_0009.png Luminal A

✓ DUAL-PATH TESTING COMPLETE!
✓ Used 512 crops from: organized_data/test_crops_512_ov128
✓ Used 144 crops from: test_crops_144_ov64
✓ Submission saved: models_dual_path/submission.csv
✓ Total predictions: 477


# Compare submission

In [15]:
import os

# Define the directory
test_dir = 'organized_data/test_crops_512_ov128'

# Get all files
files = [f for f in os.listdir(test_dir) if f.endswith('.png')]

# Rename each file
for old_name in files:
    # img_0000_crop000.png -> img_0000_000.png
    new_name = old_name.replace('_crop', '_')

    old_path = os.path.join(test_dir, old_name)
    new_path = os.path.join(test_dir, new_name)

    os.rename(old_path, new_path)